In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import os
import tempfile
import shutil

In [ ]:
rave_sim_dir = Path('/mnt/d/rave-sim-main/rave-sim-main')
simulations_dir = Path('/mnt/d/rave-sim-main/rave-sim-main/output')
print(f'rave_sim_dir: {rave_sim_dir}')

In [ ]:
sys.path.insert(0, str(rave_sim_dir / "big-wave"))

# bfpy shim — MUST be before any big-wave import
import types
try:
    import bfpy
    print('bfpy found')
except ImportError:
    _b = types.ModuleType('bfpy')
    class _CE:
        def __init__(self,*a,**kw): pass
        def position(self): return 0
        def read_chunk_c16(self,b): return len(b)
        def read_chunk_c8(self,b): return len(b)
        def write_chunk_c16(self,b): pass
        def write_chunk_c8(self,b): pass
        def advance(self,n): pass
        def __len__(self): return 0
        def dtype(self): return 'c16'
    def _nop(*a,**kw): pass
    _b.ChunkedEditor = _CE
    for m in ['generate_header_c16','generate_header_c8',
              'fft_c16','fft_c8','ifft_c16','ifft_c8']:
        setattr(_b, m, _nop)
    sys.modules['bfpy'] = _b
    print('bfpy shim installed')

import multisim
import config
import util
import propagation
from plasma import plasma_delta_beta

### 1. Physics check + generate grids

In [ ]:
energy = 8000.0; wl = propagation.convert_energy_wavelength(energy)
N = 2**18; dx = propagation.max_dx(0.01, 0.0, N, wl)

# Strong plasma: 100 layers, 5e22 cm⁻³ → phase ~3.6 rad, absorption ~5%
nz, nx = 100, N // 8
Z, Z_star, T_e, ne_pk = 13, 8.0, 100.0, 5.0e22

x_c = np.arange(nx) - nx // 2
prof = np.exp(-(x_c**2) / (2 * (nx/8)**2))
ne = np.tile((ne_pk * prof + 1e17).reshape(1, nx), (nz, 1))
ni = ne / Z_star; te = np.full((nz, nx), T_e); zs = np.full((nz, nx), Z_star)

# Vacuum control (same shape, ne=0)
ne_v = np.zeros((nz, nx)); ni_v = np.zeros((nz, nx))
te_v = np.ones((nz, nx)); zs_v = np.ones((nz, nx))

print(f'Plasma: ne_pk={ne_pk:.1e}, {nz} layers x 1um = {nz*1e-3:.2f} mm')
d_pk, b_pk, _ = plasma_delta_beta(ne_pk, ne_pk/Z_star, T_e, Z_star, Z, energy)
ph = 2*np.pi*d_pk*nz*1e-6/wl; ab = (1-np.exp(-4*np.pi*b_pk*nz*1e-6/wl))*100
print(f'Peak delta={d_pk:.4e}, beta={b_pk:.4e}')
print(f'Phase shift={ph:.3f} rad, Absorption={ab:.2f}%')

# Save grids
for name, arr in [('ne',ne),('ni',ni),('te',te),('zstar',zs)]:
    np.save(f'plasma_{name}.npy', arr)
for name, arr in [('ne',ne_v),('ni',ni_v),('te',te_v),('zstar',zs_v)]:
    np.save(f'vac_{name}.npy', arr)
print('Grids saved to .npy')

### 2. Run both simulations (plasma + vacuum)

In [ ]:
def run_one(grid_prefix, label):
    cfg = {
        'sim_params': {'N':N,'dx':dx,'z_detector':0.15,
            'detector_size':nx*dx,'detector_pixel_size_x':dx*4,
            'detector_pixel_size_y':1.,'chunk_size':4096},
        'use_disk_vector':False,'save_final_u_vectors':False,'dtype':'c8',
        'multisource':{'type':'points','energy_range':[7950,8050],
            'x_range':[0,0],'z':0.,'nr_source_points':1,'seed':1},
        'elements':[{'type':'plasma_sample','z_start':0.01,
            'pixel_size_x':dx,'pixel_size_z':1e-6,
            'ne_grid_path':f'{grid_prefix}_ne.npy',
            'ni_grid_path':f'{grid_prefix}_ni.npy',
            'te_grid_path':f'{grid_prefix}_te.npy',
            'zstar_grid_path':f'{grid_prefix}_zstar.npy',
            'Z':Z,'x_positions':[0.0]}],
    }
    sp = multisim.setup_simulation(cfg, Path.cwd(), simulations_dir)
    sc = Path(tempfile.mkdtemp())
    multisim.run_single_simulation(sp, 0, sc)
    det = np.load(sp/'00000000'/'detected.npy')[0]
    shutil.rmtree(sc)
    print(f'{label}: done, shape={det.shape}')
    return det

det_p = run_one('plasma', 'Plasma')
det_v = run_one('vac', 'Vacuum')
print('Both simulations done')

### 3. Compare results

In [ ]:
x_det_um = (np.arange(len(det_p)) - len(det_p)/2) * dx * 4 * 1e6
x_um = (np.arange(nx) - nx//2) * dx * 1e6

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Top left: direct comparison
ax = axes[0,0]
ax.plot(x_det_um, det_p, label='Plasma (5e22, 100um)', lw=1.5)
ax.plot(x_det_um, det_v, '--', label='Vacuum (ne=0, same 100 steps)', alpha=0.8)
ax.set_xlabel('x (um)'); ax.set_ylabel('Intensity')
ax.set_title('Detector: Plasma vs Vacuum (same # propagations)')
ax.legend(); ax.grid(True, alpha=0.3)

# Top right: ratio (transmission)
ax = axes[0,1]
ratio = det_p / det_v
ax.plot(x_det_um, ratio)
ax.axhline(1, color='k', ls='--', alpha=0.3)
ax.set_xlabel('x (um)'); ax.set_ylabel('Transmission (plasma/vacuum)')
ax.set_title('Transmission'); ax.grid(True, alpha=0.3)

# Bottom left: zoomed central region
ax = axes[1,0]
mask = np.abs(x_det_um) < 20
ax.plot(x_det_um[mask], det_p[mask], label='Plasma', lw=1.5)
ax.plot(x_det_um[mask], det_v[mask], '--', label='Vacuum', alpha=0.8)
ax.set_xlabel('x (um)'); ax.set_ylabel('Intensity')
ax.set_title('Zoomed central region'); ax.legend(); ax.grid(True, alpha=0.3)

# Bottom right: density profile
ax = axes[1,0].twinx() if False else axes[1,1]
ax2 = axes[1,1]
ax2.plot(x_um, ne[nz//2,:]/1e22)
ax2.set_xlabel('x (um)'); ax2.set_ylabel('n_e (10^22 cm⁻³)')
ax2.set_title('Electron density profile'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plasma_1d_comparison.png', dpi=150)
plt.show()

# Quantitative check
print(f'|det_p - det_v| max = {np.max(np.abs(det_p-det_v)):.4e}')
print(f'Transmission at centre: {ratio[len(det_p)//2]:.4f}')
print(f'Expected absorption: {ab:.2f}%')

In [ ]:
from contextlib import redirect_stdout
with open('output_plasma_1d.txt', 'w') as f:
    with redirect_stdout(f):
        for v in det_p:
            print(v)
print(f'output_plasma_1d.txt: {len(det_p)} values')

In [ ]:
print('='*50)
print('  1D PlasmaSample Test — Strong Plasma')
print('='*50)
print(f'  ne_pk = {ne_pk:.1e} cm⁻³, {nz} um thickness')
print(f'  Phase shift = {ph:.3f} rad (target: > 1 rad)')
print(f'  Absorption = {ab:.2f}% (target: > 1%)')
print(f'  |plasma - vacuum| = {np.max(np.abs(det_p-det_v)):.4e}')
print()
if np.max(np.abs(det_p-det_v)) / np.max(det_v) > 0.01:
    print('  VERDICT: Plasma effect clearly visible ✓')
else:
    print('  VERDICT: Effect too weak, need stronger parameters')